In [1]:
from pydantic import BaseModel, Field, ValidationError

class Model(BaseModel):
    number: int = Field(gt=0, lt=5)

In [2]:
from typing import Annotated

BoundedInt = Annotated[int, Field(gt=0, lt=5)]

class Model(BaseModel):
    number: BoundedInt

In [3]:
from datetime import datetime
from typing import Any

from dateutil.parser import parse

def parse_datetime(value: Any):
    if isinstance(value, str):
        try:
            return parse(value)
        except Exception as ex:
            raise ValueError(str(ex))
    return value

In [4]:
from pydantic import BeforeValidator

In [5]:
DateTime = Annotated[datetime, BeforeValidator(parse_datetime)]

In [6]:
class Model(BaseModel):
    dt: DateTime

In [7]:
Model(dt="2020/1/1 3pm")

Model(dt=datetime.datetime(2020, 1, 1, 15, 0))

In [8]:
from pydantic import AfterValidator

In [9]:
import pytz

def make_utc(dt: datetime) -> datetime:
    if dt.tzinfo is None:
        dt = pytz.utc.localize(dt)
    else:
        dt = dt.astimezone(pytz.utc)
    return dt

In [10]:
DateTimeUTC = Annotated[datetime, BeforeValidator(parse_datetime), AfterValidator(make_utc)]

In [11]:
class Model(BaseModel):
    dt: DateTimeUTC

In [12]:
Model(dt="2020/1/1 3pm")

Model(dt=datetime.datetime(2020, 1, 1, 15, 0, tzinfo=<UTC>))

In [13]:
eastern = pytz.timezone('US/Eastern')
dt = eastern.localize(datetime(2020, 1, 1, 3, 0, 0))

Model(dt=dt)

Model(dt=datetime.datetime(2020, 1, 1, 8, 0, tzinfo=<UTC>))

In [14]:
def before_validator_1(value):
    print("before_validator_1")
    return value

def before_validator_2(value):
    print("before_validator_2")
    return value
    
def before_validator_3(value):
    print("before_validator_3")
    return value

def after_validator_1(value):
    print("after_validator_1")
    return value

def after_validator_2(value):
    print("after_validator_2")
    return value

def after_validator_3(value):
    print("after_validator_3")
    return value

In [15]:
CustomType = Annotated[
    int, 
    BeforeValidator(before_validator_1),
    AfterValidator(after_validator_1),
    BeforeValidator(before_validator_2),
    AfterValidator(after_validator_2),
    AfterValidator(after_validator_3),
    BeforeValidator(before_validator_3),
]

In [16]:
class Model(BaseModel):
    number: CustomType


In [17]:
Model(number=10)

before_validator_3
before_validator_2
before_validator_1
after_validator_1
after_validator_2
after_validator_3


Model(number=10)

In [18]:
def are_elements_unique(values: list[Any]) -> list[Any]:
    unique_elements = []
    for value in values:
        if value in unique_elements:
            raise ValueError("elements must be unique")
        unique_elements.append(value)
    return values

In [19]:
len(set([1, 2, 3, 4, 5])) == len([1, 2, 3, 4, 5])

True

In [20]:
len(set([1, 1, 3, 4, 5])) == len([1, 1, 3, 4, 5])

False

In [21]:
UniqueIntegerList = Annotated[list[int], AfterValidator(are_elements_unique)]

In [22]:
class Model(BaseModel):
    numbers: UniqueIntegerList = []